In [ ]:
# import libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    """Find repo root by walking upward until expected project markers are found."""
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "config").exists() and (path / "notebooks").exists() and (path / "src").exists():
            return path
    raise RuntimeError("Could not find repo root. Launch notebook from inside the repository root.")

REPO_ROOT = find_repo_root()

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PREPROCESSING_DATA_DIR = REPO_ROOT / "data" / "processed" / "preprocessing"
OBJ2_DATA_DIR = REPO_ROOT / "data" / "processed" / "objective2_demand"
OBJ2_OUTPUT_DIR = REPO_ROOT / "outputs" / "objective2_demand"
OBJ2_MODEL_DIR = OBJ2_OUTPUT_DIR / "models"
OBJ2_VALIDATION_DIR = OBJ2_OUTPUT_DIR / "validation"
CONFIG_DIR = REPO_ROOT / "config"
CALENDAR_DIR = CONFIG_DIR / "calendars"
LOOKUP_DIR = CONFIG_DIR / "lookups"

for path in [OBJ2_DATA_DIR, OBJ2_OUTPUT_DIR, OBJ2_MODEL_DIR, OBJ2_VALIDATION_DIR]:
    path.mkdir(parents=True, exist_ok=True)


In [ ]:
# load data

weather = pd.read_parquet(OBJ2_DATA_DIR / "weather_future_features_daily.parquet")
UKCP18_LOOKUP_PATH = LOOKUP_DIR / "ukcp18_member_lookup.csv"
if not UKCP18_LOOKUP_PATH.exists():
    UKCP18_LOOKUP_PATH = PREPROCESSING_DATA_DIR / "ukcp18_member_lookup.csv"

lookup = pd.read_csv(UKCP18_LOOKUP_PATH)


In [ ]:
# Basic input checks
print("Weather shape:", weather.shape)
print("Lookup shape:", lookup.shape)

print("\nWeather columns:")
print(weather.columns.tolist())

print("\nLookup columns:")
print(lookup.columns.tolist())


In [ ]:
# show weather data
weather.head()

In [ ]:
# show weather data
lookup.head()

In [ ]:
#  Parse dates
weather["date"] = pd.to_datetime(weather["date"])


In [ ]:
# Restrict to Task 4 summary window: 2030-01-01 to 2045-12-31
start_date = pd.Timestamp("2030-01-01")
end_date = pd.Timestamp("2045-12-31")

weather_filt = weather.loc[
    (weather["date"] >= start_date) & (weather["date"] <= end_date)
].copy()

print("\nFiltered weather shape:", weather_filt.shape)
print("Filtered date range:", weather_filt["date"].min(), "to", weather_filt["date"].max())


In [ ]:
# QA checks for weather_filt dataset at a Member-level 
print("\n================ MEMBER-LEVEL QA CHECKS ================\n")

qa_summary_list = []

for member, df_member in weather_filt.groupby("climate_member"):
    
    # Row count
    row_count = len(df_member)
    
    # Unique dates
    unique_dates = df_member["date"].nunique()
    
    # Duplicate check (date + member should be unique)
    duplicate_rows = df_member.duplicated(subset=["date", "climate_member"]).sum()
    
    # Missing values per key columns
    missing_tmean = df_member["tmean_gb_c"].isna().sum()
    missing_hdd = df_member["hdd"].isna().sum()
    missing_cdd = df_member["cdd"].isna().sum()
    
    # Store results
    qa_summary_list.append({
        "climate_member": member,
        "row_count": row_count,
        "unique_dates": unique_dates,
        "duplicate_rows": duplicate_rows,
        "missing_tmean_gb_c": missing_tmean,
        "missing_hdd": missing_hdd,
        "missing_cdd": missing_cdd
    })

# Convert to DataFrame
qa_summary_df = pd.DataFrame(qa_summary_list)

# Display results
print(qa_summary_df.sort_values("climate_member"))

In [ ]:
# Add helper columns for season logic and year

weather_filt["year"] = weather_filt["date"].dt.year
weather_filt["month"] = weather_filt["date"].dt.month

# Standard meteorological seasons
# Winter = Dec, Jan, Feb
# Summer = Jun, Jul, Aug
weather_filt["is_winter"] = weather_filt["month"].isin([12, 1, 2])
weather_filt["is_summer"] = weather_filt["month"].isin([6, 7, 8])



1. Helper time-based features were derived from the date column to support seasonal aggregation. The year variable enables grouping for annual HDD/CDD calculations,
2.  while month is used to classify observations into meteorological seasons.
3.  Standard meteorological definitions were adopted (winter: December–February, summer: June–August) as they align with established climatological practice and provide a consistent basis for comparing seasonal temperature patterns across climate members.

In [ ]:
# Main member summary

member_summary = (
    weather_filt.groupby("climate_member", as_index=False)
    .agg(
        mean_tmean_gb_c=("tmean_gb_c", "mean"),
        median_tmean_gb_c=("tmean_gb_c", "median")
    )
)

A member-level annual temperature summary was created by aggregating daily tmean_gb_c values for each climate_member over the 2030–2045 analysis period.

Both the mean and median were retained: the mean provides the primary basis for ranking members from cooler to warmer, while the median offers a robustness check against any influence from unusually warm or cold days.

In [ ]:
print(member_summary)

In [ ]:
#  Winter mean temperature by member

winter_summary = (
    weather_filt.loc[weather_filt["is_winter"]]
    .groupby("climate_member", as_index=False)
    .agg(
        winter_mean_tmean_gb_c=("tmean_gb_c", "mean")
    )
)


Seasonal temperature behaviour was captured by computing the mean winter temperature for each climate member, using only observations from meteorological winter (December–February). This provides insight into cold-period conditions, which are particularly relevant for heating-driven energy demand and help differentiate members beyond annual averages.

In [ ]:
print(winter_summary)

In [ ]:
#  Summer mean temperature by member
summer_summary = (
    weather_filt.loc[weather_filt["is_summer"]]
    .groupby("climate_member", as_index=False)
    .agg(
        summer_mean_tmean_gb_c=("tmean_gb_c", "mean")
    )
)


Mean summer temperature was computed for each climate member using observations from meteorological summer (June–August). This captures warm-period conditions, which are important for assessing cooling-related demand and helps distinguish warmer climate members beyond annual averages.

In [ ]:
print(summer_summary)

In [ ]:
# Annual HDD/CDD calculation


# First calculate annual HDD/CDD totals for each member-year,
# then average across years to get mean annual HDD/CDD.

annual_hdd_cdd = (
    weather_filt.groupby(["climate_member", "year"], as_index=False)
    .agg(
        annual_hdd=("hdd", "sum"),
        annual_cdd=("cdd", "sum")
    )
)



In [ ]:
print(annual_hdd_cdd)

In [ ]:
annual_summary = (
    annual_hdd_cdd.groupby("climate_member", as_index=False)
    .agg(
        mean_annual_hdd=("annual_hdd", "mean"),
        mean_annual_cdd=("annual_cdd", "mean")
    )
)


In [ ]:
print(annual_summary)

Annual heating and cooling demand proxies were derived by first aggregating daily hdd and cdd values into yearly totals for each climate member. These annual totals were then averaged across all years in the 2030–2045 period to obtain mean_annual_hdd and mean_annual_cdd. This two-step approach ensures that the metrics reflect typical annual demand conditions, rather than being biased by daily variability, and aligns with standard practice where HDD and CDD are interpreted as annual aggregates.

In [ ]:
# Merge all summary pieces

member_summary = (
    member_summary
    .merge(winter_summary, on="climate_member", how="left")
    .merge(summer_summary, on="climate_member", how="left")
    .merge(annual_summary, on="climate_member", how="left")
    .merge(lookup[["climate_member", "member_number"]], on="climate_member", how="left")
)


In [ ]:
print(member_summary)

In [ ]:
#  Rank members from cooler to warmer
member_summary = member_summary.sort_values(
    by=["mean_tmean_gb_c", "member_number"],
    ascending=[True, True]
).reset_index(drop=True)

member_summary["temperature_rank"] = np.arange(1, len(member_summary) + 1)



In [ ]:
print(member_summary)

In [ ]:
# ------------------------------------------------------------
# Select representative members from the ranked summary
# ------------------------------------------------------------

# Count how many climate members are available in total
n_members = len(member_summary)

# Because member_summary has already been sorted from coolest to warmest
# using mean_tmean_gb_c:
# - index 0 = coolest member
# - last index = warmest member
# - middle index = median-ranked member
cool_idx = 0
warm_idx = n_members - 1
median_idx = n_members // 2

# Create a unique sorted list of the selected positions.
# Using set() avoids duplication in case the number of members is very small.
selected_indices = sorted(set([cool_idx, median_idx, warm_idx]))

# Extract only the selected members from the ranked summary
selected_members = member_summary.iloc[selected_indices].copy()

# ------------------------------------------------------------
# Assign a role to each selected member
# ------------------------------------------------------------

# Create a mapping from row index position to selection role
role_map = {}

# The first ranked member represents the cool end of the range
role_map[cool_idx] = "cool"

# The last ranked member represents the warm end of the range
role_map[warm_idx] = "warm"

# Assign the median role only if it is different from cool and warm
# (this avoids duplication when the total number of members is very small)
if median_idx not in [cool_idx, warm_idx]:
    role_map[median_idx] = "median"

# Add the selection role to each chosen member
selected_members["selection_role"] = selected_members.index.map(role_map)


# Add a plain-language explanation for why each member was selected
selected_members["selection_basis"] = selected_members["selection_role"].map({
    "cool": "Lowest mean_tmean_gb_c across members over 2030-2045",
    "median": "Median-ranked member by mean_tmean_gb_c over 2030-2045",
    "warm": "Highest mean_tmean_gb_c across members over 2030-2045"
})

# Mark these members as selected
selected_members["selected_flag"] = 1

# Keep only the key columns needed for the final selection output
selection_output = selected_members[
    [
        "climate_member",
        "member_number",
        "temperature_rank",
        "selection_role",
        "selection_basis",
        "selected_flag",
        "mean_tmean_gb_c",
        "median_tmean_gb_c",
        "winter_mean_tmean_gb_c",
        "summer_mean_tmean_gb_c",
        "mean_annual_hdd",
        "mean_annual_cdd",
    ]
].copy()

In [ ]:
selection_output.head()

In [ ]:
# Add selected flag to full summary

member_summary["selected_flag"] = member_summary["climate_member"].isin(
    selection_output["climate_member"]
).astype(int)


In [ ]:
member_summary.head(20)

In [ ]:
output_summary_fp = OBJ2_VALIDATION_DIR / "ukcp18_member_summary_2030_2045.csv"
output_selection_fp = OBJ2_DATA_DIR / "ukcp18_member_selection.csv"


In [ ]:
#  Export outputs
member_summary.to_csv(output_summary_fp, index=False)
selection_output.to_csv(output_selection_fp, index=False)

print("\nSaved:")
print("-", output_summary_fp)
print("-", output_selection_fp)

In [ ]:
# Preview outputs
print("\nMember summary preview:")
print(member_summary.head(12))

print("\nSelected members:")
print(selection_output)

In [ ]:
# ------------------------------------------------------------
# Write Task 4 README file
# ------------------------------------------------------------

readme_text = """# ukcp18_member_selection_note_README.md

## Purpose

This task identifies a small set of representative UKCP18 climate members to be used in downstream demand modelling. As the full ensemble contains multiple plausible future climate realisations, a reduced subset is selected to capture the range of temperature-driven demand outcomes while maintaining computational efficiency.

---

## Input data

Two input datasets were used:

- `weather_future_features_daily.parquet`  
  Contains daily GB-aggregated future weather variables for each UKCP18 climate member, including temperature (`tmean_gb_c`) and demand-relevant indicators (`hdd`, `cdd`).

- `ukcp18_member_lookup`  
  Provides mapping between `climate_member` identifiers and member metadata.

The analysis period was restricted to **2030–2045**, consistent with the task requirement.

---

## Data preparation and QA

The `date` column was parsed to datetime and filtered to the target period. Quality assurance checks were performed at the climate-member level to ensure:
- consistent row counts across members  
- no duplicate `(date, climate_member)` combinations  
- no missing values in key variables (`tmean_gb_c`, `hdd`, `cdd`)  
- complete temporal coverage for each member  

This ensures that all members are directly comparable.

---

## Feature engineering

Time-based features were derived to support aggregation:

- `year` enables annual grouping for HDD/CDD calculations  
- `month` supports seasonal classification  

Meteorological seasons were used:
- Winter: December–February  
- Summer: June–August  

This approach follows standard climatological practice and ensures consistent seasonal comparisons.

---

## Member-level summarisation

Each climate member was summarised using temperature and demand-relevant metrics:

### Annual temperature
- `mean_tmean_gb_c`: primary metric used for ranking members  
- `median_tmean_gb_c`: included as a robustness check against extreme values  

### Seasonal temperature
- `winter_mean_tmean_gb_c`: captures cold-period conditions relevant to heating demand  
- `summer_mean_tmean_gb_c`: captures warm-period conditions relevant to cooling demand  

Including seasonal metrics ensures that members are not only distinguished by annual averages but also by their seasonal characteristics.

### HDD/CDD aggregation
Daily `hdd` and `cdd` values were:
1. summed within each `(climate_member, year)` to obtain annual totals  
2. averaged across all years to compute `mean_annual_hdd` and `mean_annual_cdd`  

This two-step approach aligns with standard practice, as HDD and CDD are typically interpreted at the annual level.

---

## Ranking and selection logic

Climate members were ranked from cooler to warmer based on `mean_tmean_gb_c`, with `member_number` used as a secondary tie-breaker.

To represent the full range of plausible future climates, three members were selected:

- **Cool member**: lowest mean temperature  
- **Median member**: central member in the ranked distribution  
- **Warm member**: highest mean temperature  

This selection captures lower, central, and upper bounds of temperature-driven demand scenarios, ensuring that downstream modelling reflects uncertainty in future climate conditions.

---

## Outputs

Two output files were generated:

### 1. `ukcp18_member_summary_2030_2045.csv`
Contains one row per climate member with:
- temperature metrics (mean, median, seasonal)  
- demand indicators (mean annual HDD/CDD)  
- temperature rank  
- selection flag  

### 2. `ukcp18_member_selection.csv`
Contains only the selected members with:
- `selection_role` (cool / median / warm)  
- `selection_basis`  
- key summary metrics  

---

## Summary

This approach provides a transparent and statistically grounded method for reducing the UKCP18 ensemble into a manageable subset while preserving the key variation in future temperature and demand-relevant conditions.
"""

# Write to file
readme_path = OBJ2_VALIDATION_DIR / "ukcp18_member_selection_note_README.md"
readme_path.parent.mkdir(parents=True, exist_ok=True)
readme_path.write_text(readme_text, encoding="utf-8")

print(f"README file created: {readme_path}")